# **Web-Scraping---Cinema-Dataset: Top 250 films - Metadata**
## Popularity vs. Prestige: A dataset on the Top 250 films and the Academy Awards

Using web scraping, a dataset is generated detailing the 250 best films of all time, complete with technical metadata, audience metrics and their track record at the Oscars.

As this ranking is calculated using a weighted average that reflects audience ratings, the aim is to analyse whether this popularity among users correlates directly with success at the Oscars, awarded by the Academy of Motion Picture Arts and Sciences (AMPAS).

To this end, the dataset incorporates key metadata for each film (title, year, director and screenplay) alongside the volume of reviews—from both users and experts—its history of Academy Award nominations and wins, and its global box office takings. This data is extracted using web scraping techniques involving Selenium and BeautifulSoup.

## Data Extraction Methodology (Web Scraping)
*   Level 1: Main list (IMDB Top 250):
    *   Identification of the 250 films and extraction of their URLs.
*   Level 2: Film Details and Number of Reviews (Film Page):
    *   Extraction of metadata (director, screenplay, year), global box office takings and number of user reviews.
*   Level 3: Oscars Awards and Nominations (Awards section):
    *   Deep crawling to extract Oscar nominations and awards.

## We import the libraries and configure the Chrome browser to access the movie database website

In [ ]:
# The Selenium and WebDriver libraries are imported in order to access the IMDb website
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By

# We import BeautifulSoup to extract data
from bs4 import BeautifulSoup

# We import 'time' and 'robotparser' to ensure responsible web scraping
import time
import random
import urllib.robotparser

# We import pandas to create the dataset
import pandas as pd

## First, we check whether web scraping is possible on the IMDb website by reading the robots.txt file

In [ ]:
# We request the robots.txt file and read it using robotparser
robots_url = "https://www.imdb.com/robots.txt"

rp = urllib.robotparser.RobotFileParser()
rp.set_url(robots_url)
rp.read()

# We define an updated User-Agent to ensure responsible use of web scraping
user_agent = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"

# We enter the URL and check whether web scraping is possible
url = "https://www.imdb.com/chart/top/"

can_scrape = rp.can_fetch(user_agent, url)

print("¿Se puede hacer web scraping?:", can_scrape)

¿Se puede hacer web scraping?: True


In [ ]:
# We configure the Selenium WebDriver
options = webdriver.ChromeOptions()
options.add_argument("--start-maximized")

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

# We display the User-Agent we are using
print(user_agent)

Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36


In [ ]:
# We've launched the website
driver.get(url)

# We wait for a random amount of time to ensure we use web scraping responsibly and allow the webpage to load
time.sleep(random.uniform(2, 5))

# We show whether the page has loaded
print(driver.title)

IMDb Top 250 movies


## Level 1: Main list of Top 250 films
The homepage provides basic information about the list of the top 250 films; we extract the titles, year, ranking, running time, age rating and the number of user votes on IMDb

In [ ]:
# We select the information using `fin_elements`
movies = driver.find_elements(By.CSS_SELECTOR, "li.ipc-metadata-list-summary-item")

# We create lists to store each piece of data
titles = []
year = []
ratings = []
duration = []
rating_age = []
vote_count = []


# We extract the data
for movie in movies:
    try:
        title = movie.find_element(By.CSS_SELECTOR, "h3.ipc-title__text").text
        rating = movie.find_element(By.CSS_SELECTOR, ".ipc-rating-star--rating").text
        vote = movie.find_element(By.CSS_SELECTOR, ".ipc-rating-star--voteCount").text

        titles.append(title)
        ratings.append(rating)
        vote_count.append(vote)
        
        # In this case, the year, duration and recommended age are listed together,
        # so we will have to iterate through these three
        metadata = movie.find_element(By.CSS_SELECTOR, "ul.ipc-inline-list--show-dividers")
        items = metadata.find_elements(By.CSS_SELECTOR, "span.cli-title-metadata-item")
        
        
        # There are cases where one of the three pieces of data is missing, so we set it to `None`    
        if len(items) >= 3:
            year.append(items[0].text)
            duration.append(items[1].text)
            rating_age.append(items[2].text)
            
        elif len(items) == 2:
            year.append(items[0].text)
            duration.append(items[1].text)
            rating_age.append(None)

        elif len(items) == 1:
            year.append(items[0].text)
            duration.append(None)
            rating_age.append(None)

        else:
            year.append(None)
            duration.append(None)
            rating_age.append(None)
                    
    except:
        continue

## Level 2: Technical Details and Number of Reviews (Film Page)
In this section, we examine each film in detail to extract further data. We are particularly interested in the number of user reviews, as well as the number of reviews by critics in magazines and newspapers listed on IMDb.

We also collect further information, such as the director, the screenwriters, the film’s country of origin, and its worldwide box office takings.

In [ ]:
# In this section, we will extract the address of each film so that we can access each one
base_url = "https://www.imdb.com"
soup = BeautifulSoup(driver.page_source, "html.parser")
movies = soup.find_all("a", class_="ipc-title-link-wrapper")

links = []
for m in movies:
    href = m.get("href")
    if href and "/title/" in href:
        links.append(base_url + href.split("?")[0])

# We remove any duplicates
links = list(dict.fromkeys(links))


data_reviews = []

# We iterate in every direction of each film
for link in links:
    driver.get(link)
    time.sleep(random.uniform(2, 5))
    soup_movie = BeautifulSoup(driver.page_source, "html.parser")

# Film title
    try:
        title = soup_movie.find("h1").get_text(strip=True)
    except:
        title = "None"


# Director
    directors = []
    writers = []
    credits = soup_movie.find_all("li", {"data-testid": "title-pc-principal-credit"})
    for credit in credits:
        label = credit.find("span", class_="ipc-metadata-list-item__label")
    
        if not label:
            continue

        label_text = label.get_text(strip=True)

        names = [a.get_text(strip=True) for a in credit.find_all("a")]

        if "Director" in label_text:
            directors = names
        elif "Writer" in label_text:
            writers = names


# Number of reviews from users and film critics
    reviews_block = soup_movie.find("ul", {"data-testid": "reviewContent-all-reviews"})

    user_reviews = None
    critic_reviews = None

    if reviews_block:
        items = reviews_block.find_all("li")

        for item in items:
            label = item.find("span", class_="label")
            score = item.find("span", class_="score")

            if label and score:
                text_label = label.get_text(strip=True)
                text_score = score.get_text(strip=True)

                if "User reviews" in text_label:
                    user_reviews = text_score
                elif "Critic reviews" in text_label:
                    critic_reviews = text_score


# Country of origin
    countries = []
    try:
        country_block = soup_movie.find("li", {"data-testid": "title-details-origin"})
        if country_block:
            country_links = country_block.find_all("a")
            countries = [c.get_text(strip=True) for c in country_links]
        if not countries:
            countries = ["None"]
    except:
        countries = ["None"]

# Global box office takings
    gross = "None"
    try:
        labels = soup_movie.find_all("span", class_="ipc-metadata-list-item__label")
        for lab in labels:
            if "Gross worldwide" in lab.get_text():
                value = lab.find_next("span", class_="ipc-metadata-list-item__list-content-item")
                if value:
                    gross = value.get_text(strip=True)
                break
    except:
        gross = "None"

# We store everything in a set of dictionaries
    data_reviews.append({
        "Title": title,
        "Countries": countries,
        "Gross_Worldwide": gross,
        "Director": directors,
        "Writers": writers,
        "User_reviews": user_reviews,
        "Critic_reviews": critic_reviews
    })


## Level 3: Oscars Awards and Nominations (Awards section)
At the third level, we navigate to another page for each film, in this case to view the total number of awards and nominations, including Oscar awards.

On this website, the problem is that not all the nominees are displayed; some of them are hidden, so you have to click the 'Show' button. We therefore use 'execute_script(arguments.click())' and wait a moment for the page to load so that we can extract the data we need.

In [ ]:

# We create two lists
Total_Awards = []
USA_Awards = []


# We’ve updated the above links so you can access their awards website.
for link in links:
    awards_url = link.split("?")[0] + "awards/"
    driver.get(awards_url)
    time.sleep(3)


# We run the button script to display all the hidden data.
    try:
        buttons = driver.find_elements(By.CLASS_NAME, "ipc-see-more__button")
    
        for btn in buttons:
            try:
                driver.execute_script("arguments[0].click();", btn)
                time.sleep(1)
            except:
                pass
    except:
        pass




# With everything in place, we extract the HTML code using BeautifulSoup    
    soup = BeautifulSoup(driver.page_source, "html.parser")


# Film title
    try:
        title = soup.find("h2", {"data-testid": "subtitle"}).get_text(strip=True)
    except:
        title = "None"

# Total awards
    try:
        total_awards = soup.find("div", {"data-testid": "awards-signpost"}).get_text(strip=True)
    except:
        total_awards = "None"

    Total_Awards.append({
        "Title": title,
        "Total_Awards": total_awards
    })


   
    
# # Oscars and nominations: we check the HTML to see if the film was nominated for an Oscar
    academy_section = soup.find("span", {"id": "ev0000003"})

# Let's create a temporary list   
    oscars_list = []

    
    
    if academy_section:
        section = academy_section.find_parent("section")
        award_items = section.find_all("li", {"data-testid": "list-item"})

        for item in award_items:

# We check whether it is a win or a nomination
            try:
                status_text = item.find(
                    "a",
                    class_="ipc-metadata-list-summary-item__t"
                ).get_text(strip=True)
            except:
                status_text = "None"

# We sort by category
            try:
                category = item.find(
                    "span",
                    class_="awardCategoryName"
                ).get_text(strip=True)
            except:
                category = "None"

# We have compiled a list of the nominees and winners
        
            try:
                name_links = item.find_all(
                    "a",
                    class_="ipc-metadata-list-summary-item__li--link"
                )
                names = [p.get_text(strip=True) for p in name_links]

            
            except:
                namess = ["None"]

# We store the data in the temporary list
            oscars_list.append({
                'Oscar':status_text,
                'Category': category,
                'Names': names
            })

# We store all the data in the USA_Awards list in dictionaries.
    USA_Awards.append({
        "Title": title,
        "Awards": "Yes" if oscars_list else 'None',
        "Oscar_Detail": oscars_list if oscars_list else 'None'
    })



## We create a dataframe and save all the data to a dataset

In [ ]:
df_IMDb_Top250 = pd.DataFrame({
    'Title': titles,
    'Year': year,
    'Duration': duration,
    'Rating_age': rating_age,
    'Rating': ratings,
    'Vote_Count': vote_count,
    'Data_reviews': data_reviews,
    'Total_Awards': Total_Awards,
    'USA_Awards': USA_Awards
})

# We save the dataset as a CSV file
df_IMDb_Top250.to_csv("imdb_Top250.csv", index=False, encoding="utf-8")